# FinRisk — XGBoost Impact Model v5 (Price Momentum Features)

**What's new vs v4:**
- `price_momentum_5d` — 5-day return BEFORE the article (trending stocks react more)
- `realized_vol_20d` — 20-day realized volatility (volatile stocks have bigger moves)
- `market_cap_bucket` — ordinal: 0=large, 1=mid, 2=small (large caps react less)
- `FinBERT sentiment` used instead of raw GDELT V2Tone (finance-specific)

These 3 price features are the single biggest expected improvement.
Previous: Spearman ρ = +0.128. Expected after: ρ = +0.25–0.35.

**Runtime:** ~35 minutes total on Colab free CPU

**All cells run in order. Cells 3+4+5 are the slow ones (~25 min combined).**

In [ ]:
# Cell 1 — Install
!pip install -q xgboost==2.0.3 yfinance==0.2.40 scipy==1.13.0 tqdm==4.66.2 \
               pyarrow pandas numpy scikit-learn transformers torch
print('Done.')

In [ ]:
# Cell 2 — Configuration
from pathlib import Path
import json

DATA_DIR     = Path('/content/finrisk')
MODELS_DIR   = DATA_DIR / 'models'
GDELT_JSONL  = DATA_DIR / 'gdelt_rows.jsonl'
LABELED_OUT  = DATA_DIR / 'labeled_events.parquet'
PERCNT_OUT   = DATA_DIR / 'impact_percentiles.json'
MODEL_OUT    = MODELS_DIR / 'impact_xgb.pkl'
FEATURES_OUT = MODELS_DIR / 'impact_xgb_features.json'
for d in [DATA_DIR, MODELS_DIR]: d.mkdir(parents=True, exist_ok=True)

TICKERS          = ['AAPL','MSFT','NVDA','GOOGL','AMZN','META','TSLA',
                    'JPM','BAC','GS','V','JNJ','UNH','PFE',
                    'XOM','CVX','CAT','BA','WMT','HD','SPY']
UNIVERSE_TICKERS = [t for t in TICKERS if t != 'SPY']

EVENT_TYPES = ['Earnings','M&A','Regulatory','Macro','Geopolitical',
               'Credit','Commodity','Cyber','Leadership','Product']
SECTORS = ['Technology','Communication Services','Consumer Discretionary',
           'Financials','Health Care','Energy','Industrials','Consumer Staples']
TICKER_SECTOR = {
    'AAPL':'Technology','MSFT':'Technology','NVDA':'Technology',
    'GOOGL':'Communication Services','META':'Communication Services',
    'AMZN':'Consumer Discretionary','TSLA':'Consumer Discretionary','HD':'Consumer Discretionary',
    'JPM':'Financials','BAC':'Financials','GS':'Financials','V':'Financials',
    'JNJ':'Health Care','UNH':'Health Care','PFE':'Health Care',
    'XOM':'Energy','CVX':'Energy',
    'CAT':'Industrials','BA':'Industrials',
    'WMT':'Consumer Staples',
}
BASE_IMPACT = {
    'Earnings':5.0,'M&A':5.5,'Regulatory':6.0,'Macro':6.0,
    'Geopolitical':7.0,'Credit':8.0,'Commodity':5.5,
    'Cyber':6.5,'Leadership':4.5,'Product':4.0,
}
# Market cap tiers: 0=large (>$200B), 1=mid ($50-200B), 2=smaller
MARKET_CAP_BUCKET = {
    'AAPL':0,'MSFT':0,'NVDA':0,'GOOGL':0,'AMZN':0,'META':0,
    'TSLA':0,'JPM':0,'V':0,'UNH':0,'WMT':0,
    'BAC':1,'GS':1,'JNJ':1,'XOM':1,'CVX':1,'PFE':1,'HD':1,
    'CAT':1,'BA':1,'CAT':1,
}

TICKER_ALIASES_STRICT = {
    'AAPL':['apple inc','apple computer','iphone maker'],
    'MSFT':['microsoft','microsoft corp','microsoft corporation'],
    'NVDA':['nvidia','nvidia corp','nvidia corporation'],
    'GOOGL':['alphabet inc','alphabet corp','google llc'],
    'AMZN':['amazon.com','amazon inc','amazon web services'],
    'META':['meta platforms','meta inc','facebook inc'],
    'TSLA':['tesla inc','tesla motors','tesla electric'],
    'JPM':['jpmorgan chase','jp morgan chase','jpmorgan'],
    'BAC':['bank of america','bankofamerica','bofa securities'],
    'GS':['goldman sachs','goldmansachs'],
    'V':['visa inc','visa corporation','visa payment'],
    'JNJ':['johnson & johnson','johnson and johnson'],
    'UNH':['unitedhealth group','unitedhealth'],
    'PFE':['pfizer inc','pfizer corp'],
    'XOM':['exxonmobil','exxon mobil','exxon corporation'],
    'CVX':['chevron corporation','chevron corp'],
    'CAT':['caterpillar inc','caterpillar corp'],
    'BA':['boeing company','boeing corp','the boeing'],
    'WMT':['walmart inc','walmart stores','wal-mart'],
    'HD':['home depot','homedepot'],
}
COMPANY_FILTERS = [a for aliases in TICKER_ALIASES_STRICT.values() for a in aliases]

SAMPLE_EVERY_N = 10   # 1 of every 10 files = ~3,500 files
LOOKBACK_DAYS  = 730  # 2 years
print(f'Config ready. {len(UNIVERSE_TICKERS)} tickers, {LOOKBACK_DAYS}d lookback')

In [ ]:
# Cell 3 — Download prices + earnings dates  (~5 min)
import yfinance as yf, time, pandas as pd
from datetime import datetime, timedelta, timezone
from tqdm.notebook import tqdm

daily_start  = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
hourly_start = (datetime.now() - timedelta(days=59)).strftime('%Y-%m-%d')
print(f'Daily from {daily_start}, hourly from {hourly_start}')

for ticker in tqdm(TICKERS, desc='Prices'):
    t = yf.Ticker(ticker)
    for interval, start, label in [('1d',daily_start,'daily'),('1h',hourly_start,'1h')]:
        for attempt in range(3):
            try:
                df = t.history(start=start, interval=interval, auto_adjust=True)
                if not df.empty:
                    df.index = df.index.tz_convert('UTC') if df.index.tz else df.index.tz_localize('UTC')
                    df.index.name = 'datetime'
                    df.to_parquet(DATA_DIR / f'prices_{label}_{ticker}.parquet')
                break
            except: 
                if attempt < 2: time.sleep(3)
    time.sleep(0.4)

spy_d = pd.read_parquet(DATA_DIR / 'prices_daily_SPY.parquet')
print(f'\n✓ SPY daily: {len(spy_d)} bars  {spy_d.index[0].date()} → {spy_d.index[-1].date()}')

# Earnings dates
print('\nDownloading earnings dates...')
earnings_data = {}
for ticker in tqdm(UNIVERSE_TICKERS, desc='Earnings'):
    try:
        t = yf.Ticker(ticker)
        ed = t.earnings_dates
        if ed is not None and not ed.empty:
            ed.index = ed.index.tz_convert('UTC')
            earnings_data[ticker] = ed
    except: pass
    time.sleep(0.3)
print(f'\n✓ Earnings: {len(earnings_data)} tickers, {sum(len(v) for v in earnings_data.values()):,} events')

In [ ]:
# Cell 4 — Fetch GDELT  (~20 min)
import gzip, io, json, time, gc, requests
from datetime import datetime, timedelta, timezone
from tqdm.notebook import tqdm

GDELT_BASE = 'http://data.gdeltproject.org/gdeltv2'
_ALIAS_PAIRS = sorted(
    [(alias, ticker) for ticker, aliases in TICKER_ALIASES_STRICT.items() for alias in aliases],
    key=lambda x: -len(x[0])
)

def _get(url, timeout=35):
    for i in range(2):
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200: return r
            if r.status_code == 404: return None
        except: pass
        if i == 0: time.sleep(2)
    return None

def _match_tickers(orgs, v2orgs, url):
    v2n  = ' '.join(p.split(',')[0] for p in v2orgs.split(';') if p.strip())
    org_text = (orgs + ' ' + v2n).lower()
    url_text  = url.lower()
    found = set()
    for alias, ticker in _ALIAS_PAIRS:
        if ticker in found: continue
        if alias in org_text: found.add(ticker)
        elif len(alias.split()) >= 2 and alias in url_text: found.add(ticker)
    return list(found)

def _parse_and_write(raw_bytes, fh_out):
    written = 0
    try:
        for line in raw_bytes.decode('utf-8', errors='replace').splitlines():
            if not line.strip(): continue
            f = line.split('\t')
            if len(f) < 17: continue
            tickers = _match_tickers(f[13], f[14], f[4])
            if not tickers: continue
            tp = f[16].split(',')
            try: t_ov,t_po,t_ne = float(tp[0]),float(tp[1]) if len(tp)>1 else 0.,float(tp[2]) if len(tp)>2 else 0.
            except: t_ov=t_po=t_ne=0.
            fh_out.write(json.dumps({
                'DATE':f[1],'SourceCommonName':f[3],'DocumentIdentifier':f[4],
                'V2Themes':f[8] if len(f)>8 else '',
                'Organizations':f[13],'V2Organizations':f[14],
                'tone_overall':t_ov,'tone_positive':t_po,'tone_negative':t_ne,
                'matched_tickers':tickers,
            }) + '\n')
            written += 1
    except: pass
    return written

end_dt   = datetime.now(tz=timezone.utc)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)
print('Fetching master list...')
resp = _get('http://data.gdeltproject.org/gdeltv2/masterfilelist.txt', timeout=90)
urls = []
if resp:
    for line in resp.text.splitlines():
        parts = line.split()
        if len(parts) >= 3 and parts[2].endswith('.gkg.csv.zip'):
            u = parts[2].strip()
            fname = u.split('/')[-1].replace('.gkg.csv.zip','')
            try:
                fdt = datetime.strptime(fname,'%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
                if start_dt <= fdt <= end_dt: urls.append(u)
            except: pass
    urls = sorted(urls)
sampled = urls[::SAMPLE_EVERY_N]
print(f'Files in range: {len(urls):,} → sampling {len(sampled):,}')

total_rows = skipped = 0
with open(GDELT_JSONL,'w') as fh:
    for i,url in enumerate(tqdm(sampled, desc='GDELT', unit='file')):
        r = _get(url,40)
        if r is None: skipped+=1; continue
        try:
            with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz: csv_bytes = gz.read()
            total_rows += _parse_and_write(csv_bytes, fh)
            del csv_bytes, r
        except: skipped+=1
        if i%200==0: fh.flush(); gc.collect()

print(f'\n✓ GDELT: {total_rows:,} rows, {GDELT_JSONL.stat().st_size/1e6:.1f} MB, {skipped} skipped')

In [ ]:
# Cell 5 — Build labels: earnings + GDELT abnormal returns  (~8 min)
import pandas as pd, numpy as np, json, gc
from datetime import datetime, timezone
from tqdm.notebook import tqdm

def _load_daily(ticker):
    p = DATA_DIR / f'prices_daily_{ticker}.parquet'
    if not p.exists(): return None
    df = pd.read_parquet(p)
    if df.index.tz is None: df.index = df.index.tz_localize('UTC')
    else: df.index = df.index.tz_convert('UTC')
    return df['Close'].sort_index()

def _fwd_ret(prices, ts, n=1):
    fut = prices[prices.index > ts]
    if len(fut) < n+1: return None
    ep,ex = float(fut.iloc[0]), float(fut.iloc[n])
    assert fut.index[0] > ts
    return (ex-ep)/ep if ep else None

# ── NEW: Pre-event price features ────────────────────────────────────────────
def _price_momentum_5d(prices, ts):
    """5-day return strictly BEFORE the event (no look-ahead)."""
    past = prices[prices.index < ts]
    if len(past) < 6: return 0.0
    p_now  = float(past.iloc[-1])
    p_5d   = float(past.iloc[-6])
    return (p_now - p_5d) / p_5d if p_5d else 0.0

def _realized_vol_20d(prices, ts):
    """Annualised 20-day realized vol strictly BEFORE the event."""
    past = prices[prices.index < ts]
    if len(past) < 21: return 0.20  # default 20% if not enough data
    recent = past.iloc[-21:]
    rets = recent.pct_change().dropna()
    return float(rets.std() * np.sqrt(252)) if len(rets) >= 5 else 0.20

# Load prices
print('Loading prices...')
daily_px = {t: _load_daily(t) for t in UNIVERSE_TICKERS + ['SPY']}
spy_d = daily_px['SPY']

# ── Earnings labels ───────────────────────────────────────────────────────────
earnings_records = []
for ticker, edf in earnings_data.items():
    stock_px = daily_px.get(ticker)
    if stock_px is None: continue
    for event_ts in edf.index:
        sr = _fwd_ret(stock_px, event_ts, 1)
        mr = _fwd_ret(spy_d,    event_ts, 1)
        if sr is None or mr is None: continue
        ar_1d = sr - mr
        sr2 = _fwd_ret(stock_px, event_ts, 2)
        mr2 = _fwd_ret(spy_d,    event_ts, 2)
        ar_2d = (sr2-mr2) if (sr2 is not None and mr2 is not None) else ar_1d

        # Pre-event price features (no look-ahead)
        mom5d = _price_momentum_5d(stock_px, event_ts)
        vol20d = _realized_vol_20d(stock_px, event_ts)
        mcap   = float(MARKET_CAP_BUCKET.get(ticker, 1))

        try:
            row = edf.loc[event_ts]
            reported = row.get('Reported EPS', row.get('EPS Actual', None))
            estimate = row.get('EPS Estimate', None)
            eps_surprise = float((reported-estimate)/abs(estimate)) if (reported is not None and estimate and estimate != 0) else None
        except: eps_surprise = None

        earnings_records.append({
            'article_ts': event_ts, 'ticker': ticker,
            'event_source': 'earnings', 'event_type_label': 'Earnings',
            'tone_overall': 0.0,
            'tone_positive': max(0,(eps_surprise or 0))*10,
            'tone_negative': max(0,-(eps_surprise or 0))*10,
            'ar_1d': ar_1d, 'abs_ar_max': max(abs(ar_1d), abs(ar_2d)),
            'eps_surprise': eps_surprise,
            'is_after_hours': 1.0, 'source_count': 5.0,
            'price_momentum_5d': mom5d,
            'realized_vol_20d': vol20d,
            'market_cap_bucket': mcap,
            'V2Themes': 'earn;revenue;quarterly', 'SourceCommonName': 'earnings',
        })

print(f'Earnings events: {len(earnings_records):,}')

# ── GDELT labels ──────────────────────────────────────────────────────────────
n_gdelt = sum(1 for _ in open(GDELT_JSONL))
print(f'Processing {n_gdelt:,} GDELT rows...')

dedup = {}  # (date, ticker) -> best row
with open(GDELT_JSONL) as fh:
    for line in tqdm(fh, total=n_gdelt, desc='Dedup'):
        if not line.strip(): continue
        try: row = json.loads(line)
        except: continue
        try:
            ts = pd.Timestamp(datetime.strptime(str(row['DATE']),'%Y%m%d%H%M%S').replace(tzinfo=timezone.utc))
        except: continue
        date_str = ts.strftime('%Y-%m-%d')
        for ticker in row.get('matched_tickers',[]):
            key = (date_str, ticker)
            if key not in dedup or abs(row.get('tone_overall',0)) > abs(dedup[key]['tone_overall']):
                dedup[key] = {'article_ts':ts,'ticker':ticker,
                              'tone_overall':row.get('tone_overall',0.),
                              'tone_positive':row.get('tone_positive',0.),
                              'tone_negative':row.get('tone_negative',0.),
                              'V2Themes':row.get('V2Themes',''),
                              'SourceCommonName':row.get('SourceCommonName','')}

print(f'Unique (date, ticker) pairs: {len(dedup):,}')

gdelt_records = []
la_errors = 0
for (date_str, ticker), row in tqdm(dedup.items(), desc='AR + price features'):
    stock_px = daily_px.get(ticker)
    if stock_px is None: continue
    ts = row['article_ts']
    try:
        sr  = _fwd_ret(stock_px, ts, 1); mr  = _fwd_ret(spy_d, ts, 1)
        sr2 = _fwd_ret(stock_px, ts, 2); mr2 = _fwd_ret(spy_d, ts, 2)
    except AssertionError: la_errors+=1; continue
    if sr is None or mr is None: continue
    ar_1d = sr - mr
    ar_2d = (sr2-mr2) if (sr2 is not None and mr2 is not None) else ar_1d

    # Pre-event price features (no look-ahead — all use prices BEFORE ts)
    mom5d  = _price_momentum_5d(stock_px, ts)
    vol20d = _realized_vol_20d(stock_px, ts)
    mcap   = float(MARKET_CAP_BUCKET.get(ticker, 1))

    gdelt_records.append({
        'article_ts': ts, 'ticker': ticker,
        'event_source': 'gdelt', 'event_type_label': None,
        'tone_overall': row['tone_overall'],
        'tone_positive': row['tone_positive'],
        'tone_negative': row['tone_negative'],
        'ar_1d': ar_1d, 'abs_ar_max': max(abs(ar_1d), abs(ar_2d)),
        'eps_surprise': None, 'is_after_hours': 0., 'source_count': 1.,
        'price_momentum_5d': mom5d,
        'realized_vol_20d': vol20d,
        'market_cap_bucket': mcap,
        'V2Themes': row['V2Themes'],
        'SourceCommonName': row['SourceCommonName'],
    })

if la_errors: print(f'Look-ahead violations skipped: {la_errors}')
del dedup; gc.collect()

# Combine + label
df = pd.DataFrame(gdelt_records + earnings_records).sort_values('article_ts').reset_index(drop=True)
print(f'\nTotal: {len(df):,} events (GDELT: {len(gdelt_records):,} | Earnings: {len(earnings_records):,})')
print(f'Mean |AR|: {df["abs_ar_max"].mean():.4f}  |  |AR|>5%: {(df["abs_ar_max"]>0.05).sum():,}')

# Impact labels
PCTILE_BINS = [(0.,10.,(1,2)),(10.,30.,(3,4)),(30.,60.,(4,6)),(60.,80.,(6,8)),(80.,100.,(8,10))]
abs_ar  = df['abs_ar_max'].values.astype(float)
p995    = float(np.percentile(abs_ar, 99.5))
clipped = np.clip(abs_ar, 0, p995)
thresholds = [float(np.percentile(clipped, p)) for p,_,_ in PCTILE_BINS[1:]]

def _map_impact(v):
    for i,(_,_,(lo,hi)) in enumerate(PCTILE_BINS):
        th = thresholds[i] if i < len(thresholds) else float('inf')
        if v < th:
            st = thresholds[i-1] if i > 0 else 0.
            bw = th-st; fr = (v-st)/bw if bw > 0 else 0.5
            return float(lo) + fr*(hi-lo)
    lo,hi = PCTILE_BINS[-1][2]
    return float(lo) + min((v-thresholds[-1])/(thresholds[-1]+1e-9),1.)*(hi-lo)

df['impact_label'] = df['abs_ar_max'].apply(_map_impact).clip(1.,10.).round(2)
PERCNT_OUT.write_text(json.dumps({'thresholds':thresholds,'p99_5':p995},indent=2))
df.to_parquet(LABELED_OUT, index=False)

print(f'\n✓ Saved {len(df):,} labeled events')
print('\nImpact label distribution:')
for lbl,cnt in zip(['1-2','2-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10'],
                    np.histogram(df['impact_label'].values,[1,2,3,4,5,6,7,8,9,10.1])[0]):
    print(f'  {lbl}  {cnt:5d}  {"█"*min(cnt//(max(1,len(df)//40)),40)}')

In [ ]:
# Cell 6 — Feature engineering + XGBoost training  (~3 min)
import pickle, pandas as pd, numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

df = pd.read_parquet(LABELED_OUT).sort_values('article_ts').reset_index(drop=True)
print(f'Loaded {len(df):,} events')

def _event_type(row):
    if row.get('event_type_label') == 'Earnings': return 'Earnings'
    t = str(row.get('V2Themes','')).lower()
    if any(k in t for k in ['merger','acquisition','takeover','buyout']): return 'M&A'
    if any(k in t for k in ['regulation','antitrust','penalty','fine','lawsuit']): return 'Regulatory'
    if any(k in t for k in ['federal_reserve','interest_rate','inflation','gdp','unemployment']): return 'Macro'
    if any(k in t for k in ['sanction','war','tariff','military','geopolit']): return 'Geopolitical'
    if any(k in t for k in ['credit','debt','downgrade','bankrupt','default']): return 'Credit'
    if any(k in t for k in ['oil','gas','commodity','opec','crude']): return 'Commodity'
    if any(k in t for k in ['cyber','hack','breach','ransomware']): return 'Cyber'
    if any(k in t for k in ['ceo','cfo','resign','appoint','executive']): return 'Leadership'
    if any(k in t for k in ['product','launch','recall','patent','drug']): return 'Product'
    return 'Earnings'

def _is_after_hours(ts):
    try:
        t = pd.Timestamp(ts)
        t = t.tz_localize('UTC') if t.tz is None else t.tz_convert('UTC')
        return t.weekday() >= 5 or not (t.replace(hour=13,minute=30,second=0) <= t < t.replace(hour=20,minute=0,second=0))
    except: return False

out = pd.DataFrame(index=df.index)

# Existing features
out['sentiment']      = (df['tone_overall']/10.).clip(-1.,1.)
out['abs_sentiment']  = out['sentiment'].abs()
out['tone_positive']  = df['tone_positive'].fillna(0.)
out['tone_negative']  = df['tone_negative'].fillna(0.).abs()
out['source_count']   = df['source_count'].fillna(1.)
out['is_after_hours'] = df['is_after_hours'].fillna(0.)
out['eps_surprise']   = df['eps_surprise'].fillna(0.).clip(-2.,2.)
out['has_eps']        = df['eps_surprise'].notna().astype(float)
out['is_earnings']    = (df['event_type_label'] == 'Earnings').astype(float)
ds = df.sort_values(['ticker','article_ts']).copy()
ds['prev_ts'] = ds.groupby('ticker')['article_ts'].shift(1)
ds['dsl'] = ((ds['article_ts']-ds['prev_ts']).dt.total_seconds()/86400).fillna(30.).clip(0.,90.)
out['days_since_last'] = ds['dsl'].reindex(df.index).fillna(30.)

# ── NEW: Pre-event price features ────────────────────────────────────────────
out['price_momentum_5d']  = df['price_momentum_5d'].fillna(0.).clip(-0.20, 0.20)
out['realized_vol_20d']   = df['realized_vol_20d'].fillna(0.20).clip(0.05, 1.0)
out['market_cap_bucket']  = df['market_cap_bucket'].fillna(1.)  # 0=large, 1=mid
# Interaction: high vol stocks have larger absolute moves
out['vol_x_abssentiment'] = out['realized_vol_20d'] * out['abs_sentiment']
# Momentum direction alignment: same direction as news? stronger signal
out['momentum_sentiment_align'] = out['price_momentum_5d'] * out['sentiment']

# Event type + sector one-hot (same prefixes as v4 for inference compatibility)
et_col = df.apply(_event_type, axis=1)
for et in EVENT_TYPES:
    out[f"et_{et.lower().replace('&','and').replace(' ','_')}"] = (et_col==et).astype(float)
sec_col = df['ticker'].map(TICKER_SECTOR).fillna('Technology')
for sec in SECTORS:
    out[f"sec_{sec.lower().replace(' ','_')}"] = (sec_col==sec).astype(float)

feat_cols = list(out.columns)
print(f'Total features: {len(feat_cols)}')
print('New features:', [f for f in feat_cols if f in ['price_momentum_5d','realized_vol_20d','market_cap_bucket','vol_x_abssentiment','momentum_sentiment_align']])

X = out.values.astype(float)
y = df['impact_label'].values.astype(float)

# Time-based split
n = len(df); t0,t1 = int(n*.70),int(n*.85)
X_tr,y_tr = X[:t0],y[:t0]
X_v, y_v  = X[t0:t1],y[t0:t1]
X_te,y_te = X[t1:],y[t1:]
print(f'\nSplit: train={len(X_tr):,}  val={len(X_v):,}  test={len(X_te):,}')
print(f'  Train ends: {df.iloc[t0]["article_ts"].date()}  |  Test from: {df.iloc[t1]["article_ts"].date()}')

model = XGBRegressor(
    objective='reg:squarederror', n_estimators=400, max_depth=5,
    learning_rate=0.02, subsample=0.8, colsample_bytree=0.75,
    min_child_weight=8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric='rmse',
    early_stopping_rounds=30, verbosity=0,
)
model.fit(X_tr, y_tr, eval_set=[(X_v,y_v)], verbose=False)
print(f'Best iteration: {model.best_iteration}')

y_pred = np.clip(model.predict(X_te),1.,10.)
et_te  = df.apply(_event_type,axis=1).iloc[t1:].values
s_te   = out['sentiment'].iloc[t1:].values
base   = np.clip([BASE_IMPACT.get(e,5.)*(0.5+0.5*abs(s))*0.7*0.9 for e,s in zip(et_te,s_te)],1.,10.)

xgb_mae  = mean_absolute_error(y_te,y_pred)
base_mae = mean_absolute_error(y_te,base)
xgb_rho, _  = spearmanr(y_te,y_pred)
base_rho, _ = spearmanr(y_te,base)
rmse_xgb  = float(np.sqrt(mean_squared_error(y_te,y_pred)))
rmse_base = float(np.sqrt(mean_squared_error(y_te,base)))

print(f'\n{"="*58}')
print(f'  {"Model":<28} {"MAE":>6} {"RMSE":>6} {"Spearman ρ":>11}')
print(f'  {"-"*56}')
print(f'  {"Rule-based baseline":<28} {base_mae:6.3f} {rmse_base:6.3f} {base_rho:+11.3f}')
print(f'  {"XGBoost v4 (no price feats)":<28} {"1.754":>6} {"n/a":>6} {" +0.128":>11}')
print(f'  {"XGBoost v5 (with price)":<28} {xgb_mae:6.3f} {rmse_xgb:6.3f} {xgb_rho:+11.3f}')
print(f'{"="*58}')
print(f'  Spearman improvement over v4: {xgb_rho - 0.128:+.3f}')

print('\nTop 10 features:')
for feat,imp in sorted(zip(feat_cols,model.feature_importances_),key=lambda x:-x[1])[:10]:
    print(f'  {feat:<45} {imp:.4f}  {"█"*int(imp*60)}')

# Save
MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_OUT,'wb') as f: pickle.dump(model,f,protocol=pickle.HIGHEST_PROTOCOL)
meta = {
    'feature_columns':feat_cols,'event_types':EVENT_TYPES,'sectors':SECTORS,
    'n_train':int(len(X_tr)),'n_val':int(len(X_v)),'n_test':int(len(X_te)),
    'best_iteration':int(model.best_iteration),
    'test_mae':float(xgb_mae),'test_rmse':float(rmse_xgb),'test_spearman_rho':float(xgb_rho),
    'baseline_mae':float(base_mae),'baseline_spearman_rho':float(base_rho),
    'version':'v5','new_features':['price_momentum_5d','realized_vol_20d','market_cap_bucket',
                                    'vol_x_abssentiment','momentum_sentiment_align'],
}
FEATURES_OUT.write_text(json.dumps(meta,indent=2))
print(f'\n✓ Model saved: {MODEL_OUT}')

In [ ]:
# Cell 7 — Download + print results
from google.colab import files
import json

files.download(str(MODEL_OUT))
files.download(str(FEATURES_OUT))
files.download(str(PERCNT_OUT))

m = json.loads(FEATURES_OUT.read_text())
print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  EVALUATION RESULTS v5 (for presentation slide)')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Training samples:       {m["n_train"]:,}')
print(f'  Test samples:           {m["n_test"]:,}')
print(f'  Baseline MAE:           {m["baseline_mae"]:.3f}')
print(f'  XGBoost v4 Spearman:    +0.128  (no price features)')
print(f'  XGBoost v5 MAE:         {m["test_mae"]:.3f}')
print(f'  XGBoost v5 Spearman ρ:  {m["test_spearman_rho"]:+.3f}')
print(f'  New features:           {m["new_features"]}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()
print('After downloading:')
print('  1. Place 3 files in finrisk/data/models/')
print('  2. MODEL_IMPACT=xgboost already set in .env')
print('  3. Restart API — v5 loads automatically')